# Train an SO101-Nexus policy with demo-seeded PPO on MuJoCo Warp

Bootstrap a policy from 10 teleop demos, then fine-tune it with reinforcement learning. Run every cell top to bottom.

## 1. Check the GPU

Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi -L

## 2. Install

The example scripts live in `examples/`, not the wheel, so this clones the repo and installs the `warp` + `train` extras.

In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip uninstall -y so101-nexus 2>/dev/null || true
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"
# `pip install -e` registers the package via a .pth file that Python only
# reads when an interpreter starts. This kernel was already running before
# the install above, so `!python ...` cells (fresh subprocesses) see the
# package but in-kernel `import so101_nexus...` cells below will not unless
# `src/` is also put on sys.path explicitly, here.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

## 3. Choose the seed

`WarpPickLift-v1` is the only environment this recipe is tuned for. Seed 5 reproduces our best result; any other seed also works.

In [ ]:
SEED = 5
STEPS = 30_000_000
print(f"Training WarpPickLift-v1 for {STEPS:,} env steps, seed={SEED}")

## 4. Launch TensorBoard

Run before training so the dashboard picks up the new run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

Runs demo-seeded PPO for 30M steps. Takes a while, so grab a coffee.

In [ ]:
!python examples/bc_ppo_warp.py --exp-name colab --seed {SEED} \
    --total-timesteps {STEPS} --eval-freq 0

## 6. Evaluate the trained policy

Runs a deterministic evaluation across 512 parallel Warp environments.

In [ ]:
CKPT = "runs/WarpPickLift-v1__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id WarpPickLift-v1 --checkpoint "{CKPT}"

## 7. Watch a sample rollout

Renders one episode of the trained policy so you can see it work.


In [ ]:
# Render one deterministic rollout of the trained policy and play it inline.
import glob

from IPython.display import Video

from examples.bc_ppo_warp import rollout_video_from_checkpoint

ENV_ID = "WarpPickLift-v1"
checkpoint = sorted(glob.glob(CKPT))[-1]
print(f"Rendering sample rollout from {checkpoint}")

metrics, video_path = rollout_video_from_checkpoint(
    checkpoint,
    ENV_ID,
    control_mode="pd_joint_delta_pos",
    episode_length=512,
    hidden_dim=256,
    seed=12345,
)
print(
    f"rollout success_rate={metrics['eval/success_rate']:.2f} "
    f"return={metrics['eval/return']:.2f} "
    f"ep_len={metrics['eval/ep_len']:.0f}"
)
Video(video_path)

## Next steps

- Compare against vanilla PPO by rerunning with `--use-demos false`.
- Full walkthrough: [Training with PPO](https://so101-nexus.com/docs/training/ppo).